Data to 3d CNN
some 3d cnn implementations
https://github.com/myubabe/3D-CNNs-Model-Hub

ResNet in tensorflow
https://www.analyticsvidhya.com/blog/2021/08/how-to-code-your-resnet-from-scratch-in-tensorflow/#:~:text=This%20article%20is%20a%20step-by-step%20process%20of%20learning,the%20vanishing%20gradient%20problem%20in%20deep%20neural%20networks.

## Numpy to Nifty
- gets the numpy file and converts it into a nifty gz file for each class

In [ ]:
import nibabel as nib
import tensorflow as tf
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path
from scipy import ndimage

# the main classes for the multiclass classification
CLASSES = ["FDG","CHOLIN","DOPA"]

IN_PATH = "PATH_TO_NUMPY_ARRAYS_OF_THE_STUDIES"
STAGE1_PATH = "FOLDER_FOR_STAGE1_EXPORT"
STAGE2_PATH = "FOLDER_FOR_STAGE2_EXPORT"
# here all studies exported to niffty format
STAGE3_PATH = "FOLDER_FOR_STAGE3_EXPORT"

# how much instances can be removed
REMOVE_INSTANCES = 10 # five from start five from end

def normalize_for_cnn(volume):
    """Normalizes the data for CNN 

    Args:
        volume (np.array): the volume of the data

    Returns:
        np.array: normalized data
    """
    volume = (volume - np.min(volume)) / (np.max(volume) - np.min(volume))
    return volume.astype(np.float32)

def rotate(volume):
    """Rotate the volume by a few degrees"""

    def scipy_rotate(volume):
        # define some rotation angles
        angles = [-10, -5, 5, 10]
        # pick angles at random
        angle = random.choice(angles)
        print(f"rotated by {angle}")
        # rotate volume
        volume = ndimage.rotate(volume, angle, reshape=False)
        #ndimage.rotate(volume, angle, reshape=False)
        return volume

    augmented_volume = scipy_rotate(volume)
    #augmented_volume.set_shape(volume.shape)
    return augmented_volume

def remove_five_start_end(data):
    """Deletes 5 from start and 5 from end
    and returns the flipped array -> data start from head
    """
    end_remove_arr = np.delete(data,[0,1,2,3,4],axis=0)
    start_arr = np.flip(end_remove_arr,axis=0)
    arr = np.delete(start_arr,[0,1,2,3,4],axis=0)
    return arr

def plot_slices(num_rows, num_columns, width, height, data):
    print("data",data.shape)
    """Plot a montage of 20 CT slices"""
    data = data[:,:,:,0]
    num_slices = num_rows * num_columns
    slices = np.linspace(0, data.shape[0] - 1, num_slices, dtype=int)
    data = tf.gather(data,slices)

    fig, axes = plt.subplots(num_rows, num_columns, figsize=(12, 10))
    fig.subplots_adjust(wspace=0, hspace=0, left=0, right=1, bottom=0, top=1)

    for i, slice_index in enumerate(slices):
        row = i // num_columns
        col = i % num_columns
        axes[row, col].imshow(data[i],cmap="gray")
        axes[row, col].axis("off")

    plt.show()

def create_nifty(clazz):
    """Takes the padded data flips the to head/craniocaudal and saves them as nifty zipped files into stage3 folder

    Args:
        clazz (string): class of the series
    """
    index = 1
    files = get_files(STAGE2_PATH,clazz)
    for file in files:
        data = np.load(f"{STAGE2_PATH}/{clazz}/{file}")
        data = np.flip(data, axis=0)
        data = normalize_for_cnn(data)
        nifti_image = nib.Nifti1Image(data, affine=np.eye(4))
        file = Path(file)
        file = file.with_suffix('')
        out_file = f"{STAGE3_PATH}/{clazz}/{file}_{index}.nii.gz"
        print("saving to file:", out_file)
        nib.save(nifti_image,out_file)
        index +=1
    print("DONE")

def get_files(path,clazz):
    class_path = f"{path}/{clazz}"
    files = os.listdir(class_path)
    return files

def get_numpy_size_from_folder_stage1(path,clazz):
    """Removes the first 5 and last 5 instances, lot of noise
    then saves this into stage1 folder with head first and also one randomly rotated data and returns the
    greatest count of instances from the clazz studies

    Args:
        path (_type_): Path where the data is located
        clazz (_type_): Clazz/Subfolder with data

    Returns:
        _type_: the greatest count
    """
    class_path = f"{path}/{clazz}"
    files = get_files(path,clazz)
    samples = len(files)
    print(f"Samples number: {samples} for class: {clazz}")
    instance_count = []
    for file in files:
        data = np.load(f"{class_path}/{file}")
        dt = remove_five_start_end(data)
        instance_count.append(dt.shape[0])
        stage1_path = f"{STAGE1_PATH}{clazz}/{file}"
        np.save(stage1_path, dt, True)
        dt_rotated = rotate(dt)
        stage1_path = f"{STAGE1_PATH}{clazz}/r_{file}"
        np.save(stage1_path,dt_rotated,True)
        
    print("Sizes of the class:", instance_count)
    max_size = np.amax(instance_count)
    return max_size
    
def create_harmonized_numpy(path,clazz,max_count):
    """Takses the series if the instance flips it again back to legs first and if 
    the count is less then the greates count of instances adds 0 so the shape af all array is the same
    data is saved then legs first caudocranial
    Args:
        path (_type_): main path with data
        clazz (_type_): clazz/subfolder with series data
        max_count (_type_): the obtained max count in all three classes
    """
    class_path = f"{path}/{clazz}"
    files = get_files(path,clazz)
    for file in files:
        data = np.load(f"{class_path}/{file}")
        data = np.flip(data, axis=0)
        if (data.shape[0] < max_count):
            print(f"{file} needs to be harmonized")
            difference = max_count - data.shape[0]
            filling = np.zeros((difference, 220,220,1))
            res = np.concatenate((filling,data),axis=0)
            harm_class_path = f"{STAGE2_PATH}/{clazz}/{file}"
            manipulated = resize(res,max_count)
            np.save(harm_class_path,manipulated,True)
        else:
            print(f"{file} does not need to be harmonized")
            harm_class_path = f"{STAGE2_PATH}/{clazz}/{file}"
            manipulated = resize(data,max_count)
            np.save(harm_class_path,manipulated,True)

def resize(volume, max_count):
    """Resizes the data to to half

    Args:
        volume (np.array): numpy array with the data
        max_count (integer): number of instances

    Returns:
        np.array: The resized volume //2
    """
    desired_width = 110
    desired_height = 110
    # this is for already precreated model with dept 299
    desired_depth = 299
    # or we can make the half
    #desired_depth = max_count // 2
    current_depth = volume.shape[0]
    current_width = volume.shape[1]
    current_height = volume.shape[2]
    volume = ndimage.zoom(volume, (desired_depth / current_depth, desired_width / current_width, desired_height / current_height, 1), order=1)
    return volume
    
def get_greatest_instances():
    my_sizes=[]
    for clazz in CLASSES:
        count_int = get_numpy_size_from_folder_stage1(IN_PATH,clazz)
        my_sizes.append(count_int)
    return np.amax(my_sizes)
    
def prepare_dataset():                          
    for clazz in CLASSES:
        path = f"{IN_PATH}/{clazz}"
        print("Looking into path:",path)
        files = os.listdir(path)
        create_nifty(files,clazz)

def stage_one_to_three():
    #stage1 
    max_data = get_greatest_instances()
    print(max_data)
    #stage 2
    for clazz in CLASSES:
        create_harmonized_numpy(STAGE1_PATH,clazz, max_data)
    print("Done") 
    #stage3
    for clazz in CLASSES:
        create_nifty(clazz)

def show_the_data(clazz,stage_path):
    files = get_files(stage_path,clazz)
    data = nib.load(f"{stage_path}{clazz}/{files[18]}")
    data = data.get_fdata()
    plot_slices(10,10,110,110,data)
    
def augment_data(clazz,stage_path):
    files = get_files(stage_path,clazz)
    for file in files:
        path_to_file = f"{stage_path}{clazz}/{file}"
        data = nib.load(path_to_file)
        data = data.get_fdata()
        data = rotate(data)
        nifti_image = nib.Nifti1Image(data, affine=np.eye(4))
        out_file = f"{stage_path}{clazz}/r_{file}"
        print("saving to file:", out_file)
        nib.save(nifti_image,out_file)
        
    
#pipeline the data
stage_one_to_three()

for clazz in CLASSES:
    create_nifty(clazz)




